# Multi-token prediction (`variant: mtp`)

<img src="../../assets/explainers/mtp.gif" width="100%">

**Training**: head $k$'s target row is the input shifted by $k+1$; a sweep shows one position feeding all 4 heads, then the per-head loss (far heads are harder). **Generation**: self-speculative decoding - each forward checks the purple drafts from heads 1-3 (green ✓ = head 0 agrees, red ✗ = replaced) and drafts again in the same pass. Watch the counter: ~3 characters per forward pass instead of 1.

## Intuition

A next-token model only ever has to think one step ahead. Asking the same hidden state to also predict
2, 3, 4 tokens ahead forces it to encode a short *plan* (Gloeckle et al., 2024). The extra heads are
then free drafters at inference: they guess a few tokens, and the ordinary next-token head checks the
guesses in a single pass.

## Math

Trunk states $h_i$; head $k \in \{0,\dots,K-1\}$ outputs $p_k(\cdot \mid h_i)$, trained on $x_{i+k+1}$:

$$\mathcal{L} = \frac{1}{K}\sum_{k=0}^{K-1} \mathrm{CE}\big(p_k(\cdot\mid h_i),\, x_{i+k+1}\big).$$

Head 0 is the usual LM head; heads $1..K-1$ are a residual MLP followed by the **shared** unembedding
(`MTPHead`), initialised to the identity so every head starts as a next-token head.

**Self-speculative decoding.** With drafts $d_1..d_m$ appended to the prefix, one forward gives head 0's
distribution at every draft position. Walk left to right: sample $s_j \sim p_0(\cdot \mid
\text{prefix}, d_{<j})$; if $s_j = d_j$ accept and continue, otherwise emit $s_j$ and stop; if all hold,
emit a bonus sample. Every emitted token is a sample from head 0 given the true prefix, so **the output
distribution is exactly autoregressive sampling of head 0** - only the number of forwards changes.

## In the code

| File | What |
|---|---|
| `models_mtp/model.py` | `MultiTokenPredictionLM` (`trunk`, `forward → [K logits]`), `MTPHead`, `collate_mtp` |
| `models_mtp/loss.py` | `head_losses`, `mtp_loss` (mean of per-head shifted CEs) |
| `models_mtp/sample.py` | `sample_mtp` (draft → verify), `SpeculativeStep`, `model.tokens_per_forward` |
| `models_mtp/metrics.py` | `head_accuracy` → `head{k}_accuracy` |

In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repo: put src/ on the path if hale_llm is not installed.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import HTML, display

import hale_llm.models  # noqa: F401  - registers every paradigm
from hale_llm.core.registry import get_loss, get_sampler, get_variant
from hale_llm.dataloader.collate import get_collate
from hale_llm.visualization.explainers import CORPUS, get_explainer

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print("training corpus:")
for sentence in CORPUS:
    print("  ", repr(sentence))

## Train a toy model

Four heads (`n_mtp_heads: 4`) on a 2-layer trunk. The explainer's recipe trains a ~100k-parameter model on the four corpus sentences
in a few seconds on a CPU - small enough to inspect every number, trained enough to show real
behaviour.

In [ ]:
explainer = get_explainer("mtp")
run = explainer.train()
tok, model, cfg = run.tokenizer, run.model, run.cfg
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("config:", {k: v for k, v in cfg.model.model_dump().items()
                  if k in ("arch", "d_model", "n_layers", "attn_type", "use_time_cond",
                           "block_size", "n_mtp_heads", "fm_schedule")})

fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.semilogy(run.losses, color="#264653")
ax.set(xlabel="step", ylabel="training loss", title="toy training")
plt.show()

## Every head on its own offset

In [ ]:
from hale_llm.models.models_mtp import head_losses

batch = get_collate("mtp")(run.ids, tok)
with torch.no_grad():
    logits = model(batch)
    losses = head_losses(model, batch)
i = 8
print(f"trunk state at position {i} ({run.text[i]!r}); heads predict:")
for k, lg in enumerate(logits):
    guess = tok.token(int(lg[0, i].argmax()))
    print(f"  head {k}: x[{i + k + 1}] -> {guess!r}  (truth {run.text[i + k + 1]!r})   loss {float(losses[k]):.4f}")

Position 8 sits right before the "model"/"robot" choice: head 0 is predicting the branch itself, heads 1-3
predict characters *inside* the not-yet-chosen word, so their guesses only make sense for one branch.
This is where drafts get rejected.

## Greedy speculative decoding equals greedy autoregressive decoding

In [ ]:
from hale_llm.models.models_mtp import sample_mtp

prompt = run.ids[:, :8]
spec = sample_mtp(model, tok, prompt, max_new_tokens=40, temperature=0.0)
x = prompt
with torch.no_grad():
    for _ in range(40):
        x = torch.cat([x, model({"input_ids": x})[0][:, -1].argmax(-1, keepdim=True)], 1)
print("speculative :", tok.decode(spec[0]))
print("one-by-one  :", tok.decode(x[0]))
print("identical   :", torch.equal(spec, x), f"| {model.tokens_per_forward:.2f} tokens per forward")

## Speed-up vs temperature and number of heads used

In [ ]:
import numpy as np

rows = []
for T in (0.0, 0.7, 1.0, 1.5):
    tpf = []
    for seed in range(12):
        torch.manual_seed(seed)
        sample_mtp(model, tok, prompt, max_new_tokens=40, temperature=T)
        tpf.append(model.tokens_per_forward)
    rows.append((T, float(np.mean(tpf))))
for T, v in rows:
    print(f"T={T:<4} mean tokens/forward = {v:.2f}   (autoregressive = 1.00, ceiling = {model.n_heads})")

In [ ]:
trace = explainer.trace(run, seed=0)
for k, s in enumerate(trace.steps):
    drafts = "".join(tok.token(d) for d in s.drafts) or "-"
    emitted = "".join(tok.token(e) for e in s.emitted)
    print(f"forward {k + 1:2d}: drafts {drafts!r:7} accepted {s.accepted}/{len(s.drafts)}  emitted {emitted!r}")

In [ ]:
# Re-render the animation from this notebook's model. The committed copy shown at the top is
# assets/explainers/mtp.gif (regenerate all of them with `hale-llm-explain`).
from PIL import Image

out = ROOT / "notebooks" / "models" / "outputs" / "mtp.gif"  # git-ignored
explainer.build(out, run=run)
with Image.open(out) as gif:
    print(out.relative_to(ROOT), f"{gif.n_frames} frames, {gif.size[0]}x{gif.size[1]},",
          f"{out.stat().st_size / 1024:.0f} KB")
# In Jupyter: display(HTML('<img src="outputs/mtp.gif" width="100%">'))

## Takeaways

* Same parameters as an AR model plus $K-1$ small heads; training adds look-ahead pressure on the trunk.
* Speculative decoding is **lossless**: identical output distribution, fewer forward passes. The speed-up
  is set by how often heads agree with head 0 - high in predictable text, low at genuine choices.
* `hale-mtp` in HaleBlocks loads an MTP checkpoint as a VLM / VLA language backbone (main head only).

**Exercises.** (1) Train with `model={"n_mtp_heads": 8}` - where does acceptance stop improving?
(2) Plot `head_accuracy` against $k$ with the `head_accuracy` metric.